In [ ]:
# 1. Imports and Configuration
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

WINDOW_SIZE = 6       # 6 observations = 30-minute window
FEATURES = 8

# 2. Feature Definition (Order is crucial for runtime input)
# Feature order:
# 0 - rainfall (mm)
# 1 - temperature (°C)
# 2 - humidity (%)
# 3 - wind_speed (km/h)
# 4 - water_level (m)
# 5 - soil_moisture (%)
# 6 - slope_tilt (degrees)
# 7 - vibration (units)


## 2. Feature Definitions and Sensor Ranges

The order of features is crucial for input consistency. The `SENSOR_RANGES` dictionary defines the valid minimum and maximum values for each feature, which will be used for input validation in the simulator.

In [ ]:
# Sensor ranges for input validation
SENSOR_RANGES = {
    "Rainfall": (0, 100),       # mm
    "Temperature": (15, 50),    # °C
    "Humidity": (10, 100),      # %
    "Wind Speed": (0, 60),      # km/h
    "Water Level": (1, 10),     # m
    "Soil Moisture": (0, 100),  # %
    "Slope/Tilt": (5, 50),      # degrees
    "Vibration": (0, 100)       # units
}

In [ ]:
# 3. Synthetic 30-Minute Sliding-Window Data Generation
# 4. Synthetic Label Generation

def generate_data(samples=1500): # Changed default samples to 1500
    X = np.zeros((samples, WINDOW_SIZE, FEATURES), dtype=np.float32)
    y_flood = np.zeros(samples, dtype=np.int32) # Changed to int32 for sparse_categorical_crossentropy
    y_fire = np.zeros(samples, dtype=np.int32)   # Changed to int32
    y_landslide = np.zeros(samples, dtype=np.int32) # Changed to int32

    # Feature ranges (min, max) - unchanged, but will be used for clipping
    RANGES = [
        (0, 100),   # Rainfall
        (15, 50),   # Temperature
        (10, 100),  # Humidity
        (0, 60),    # Wind speed
        (1, 10),    # Water level
        (0, 100),  # Soil moisture
        (5, 50),    # Slope/tilt
        (0, 100)    # Vibration
    ]

    # Hazard Score Thresholds for 3 classes (adjusted for better distribution of NORMAL/WATCH/ALERT)
    # These thresholds define when a hazard transitions from one state to another based on its synthetic score (0-100)
    FLOOD_THRESHOLDS = {'WATCH': 30, 'ALERT': 65} # Score < 30 -> NORMAL, 30-65 -> WATCH, >65 -> ALERT
    FIRE_THRESHOLDS = {'WATCH': 50, 'ALERT': 75} # Score < 50 -> NORMAL, 50-75 -> WATCH, >75 -> ALERT
    LANDSLIDE_THRESHOLDS = {'WATCH': 60, 'ALERT': 85} # Score < 60 -> NORMAL, 60-85 -> WATCH, >85 -> ALERT


    for i in range(samples):
        # Initialize start values for the window based on realistic ranges
        # Increased spread in initial values to encourage diverse scenarios
        start_values = np.array([
            np.random.uniform(RANGES[0][0], RANGES[0][1]), # Rainfall
            np.random.uniform(RANGES[1][0], RANGES[1][1]), # Temperature
            np.random.uniform(RANGES[2][0], RANGES[2][1]), # Humidity
            np.random.uniform(RANGES[3][0], RANGES[3][1]), # Wind speed
            np.random.uniform(RANGES[4][0], RANGES[4][1]), # Water level
            np.random.uniform(RANGES[5][0], RANGES[5][1]), # Soil moisture
            np.random.uniform(RANGES[6][0], RANGES[6][1]), # Slope/tilt
            np.random.uniform(RANGES[7][0], RANGES[7][1])  # Vibration
        ])

        # Decide if this sample should represent worsening conditions or stable
        # Using a probability distribution to ensure variety
        condition_type = np.random.choice(
            ['stable', 'worsening_flood', 'worsening_fire', 'worsening_landslide'],
            p=[0.40, 0.20, 0.20, 0.20]
        )

        for t in range(WINDOW_SIZE):
            for f in range(FEATURES):
                # Apply base random fluctuation
                val = start_values[f] + np.random.uniform(-1.5, 1.5)

                # Apply worsening trends based on condition_type
                if condition_type == 'worsening_flood':
                    if f == 0: val += np.random.uniform(0.8, 4.0) * t # Rainfall increase
                    if f == 4: val += np.random.uniform(0.2, 0.8) * t # Water Level increase
                    if f == 5: val += np.random.uniform(0.5, 2.5) * t # Soil Moisture increase
                elif condition_type == 'worsening_fire':
                    if f == 1: val += np.random.uniform(0.3, 1.8) * t # Temperature increase
                    if f == 2: val -= np.random.uniform(0.5, 3.0) * t # Humidity decrease
                    if f == 3: val += np.random.uniform(0.4, 2.0) * t # Wind Speed increase
                elif condition_type == 'worsening_landslide':
                    if f == 0: val += np.random.uniform(0.7, 3.5) * t # Rainfall increase
                    if f == 5: val += np.random.uniform(0.6, 3.0) * t # Soil Moisture increase
                    if f == 6: val += np.random.uniform(0.2, 1.0) * t # Slope/Tilt increase
                    if f == 7: val += np.random.uniform(0.5, 2.5) * t # Vibration increase

                # Clip values to ensure they stay within realistic ranges (from RANGES definition)
                X[i, t, f] = np.clip(val, RANGES[f][0], RANGES[f][1])

        # --------------------------------------------------------
        # Generate labels based on the entire window for the current sample
        # --------------------------------------------------------

        # Flood Label Rules
        avg_rainfall = X[i, :, 0].mean()
        avg_water_level = X[i, :, 4].mean()
        avg_soil_moisture_flood = X[i, :, 5].mean() # Added soil moisture influence
        water_level_change = X[i, WINDOW_SIZE-1, 4] - X[i, 0, 4] # Change from start to end

        # Weighted score for Flood
        flood_score = (avg_rainfall * 0.2) + (avg_water_level * 0.4) + (avg_soil_moisture_flood * 0.1) + (max(0, water_level_change) * 5)
        flood_score = np.clip(flood_score, 0, 100) # Ensure score is within 0-100

        if flood_score < FLOOD_THRESHOLDS['WATCH']:
            y_flood[i] = 0 # NORMAL
        elif flood_score < FLOOD_THRESHOLDS['ALERT']:
            y_flood[i] = 1 # WATCH
        else:
            y_flood[i] = 2 # ALERT

        # Forest Fire Label Rules
        avg_temp = X[i, :, 1].mean()
        min_humidity = X[i, :, 2].min() # Minimum humidity is critical for fire risk
        max_wind_speed = X[i, :, 3].max() # Maximum wind speed is critical
        avg_rainfall_fire = X[i, :, 0].mean() # Lower rainfall increases dryness

        # Weighted score for Fire
        fire_score = (avg_temp * 0.3) + ((100 - min_humidity) * 0.4) + (max_wind_speed * 0.2) + ((100 - avg_rainfall_fire) * 0.1) # Inverse rainfall effect
        fire_score = np.clip(fire_score, 0, 100) # Ensure score is within 0-100

        if fire_score < FIRE_THRESHOLDS['WATCH']:
            y_fire[i] = 0 # NORMAL
        elif fire_score < FIRE_THRESHOLDS['ALERT']:
            y_fire[i] = 1 # WATCH
        else:
            y_fire[i] = 2 # ALERT

        # Landslide Label Rules
        avg_soil_moisture_landslide = X[i, :, 5].mean()
        max_slope_tilt = X[i, :, 6].max()
        max_vibration = X[i, :, 7].max()
        avg_rainfall_landslide = X[i, :, 0].mean()

        # Weighted score for Landslide
        landslide_score = (avg_soil_moisture_landslide * 0.2) + (max_slope_tilt * 0.3) + (max_vibration * 0.3) + (avg_rainfall_landslide * 0.2)
        landslide_score = np.clip(landslide_score, 0, 100) # Ensure score is within 0-100

        if landslide_score < LANDSLIDE_THRESHOLDS['WATCH']:
            y_landslide[i] = 0 # NORMAL
        elif landslide_score < LANDSLIDE_THRESHOLDS['ALERT']:
            y_landslide[i] = 1 # WATCH
        else:
            y_landslide[i] = 2 # ALERT

    print("\n--- Generated Label Distribution ---")
    hazard_names = ["Flood", "Fire", "Landslide"]
    labels = [y_flood, y_fire, y_landslide]
    state_map = {0: "NORMAL", 1: "WATCH", 2: "ALERT"}

    for j, hazard_label_array in enumerate(labels):
        print(f"\n{hazard_names[j]} class distribution:")
        counts = np.bincount(hazard_label_array, minlength=3) # Ensure all 3 bins are counted
        for class_idx in range(3):
            # Check if class_idx is within the bounds of counts to prevent IndexError if no samples for a class
            if class_idx < len(counts):
                print(f"  {state_map[class_idx]}: {counts[class_idx]}")
            else:
                print(f"  {state_map[class_idx]}: 0") # No samples for this class

    return X, {
        "flood": y_flood,
        "fire": y_fire,
        "landslide": y_landslide
    }

In [ ]:
# 5. Lightweight Multi-Hazard Model
# 6. Model Compilation

def build_model():

    inputs = layers.Input(
        shape=(WINDOW_SIZE, FEATURES),
        name="environmental_input"
    )

    # Shared temporal feature extractor
    x = layers.SeparableConv1D(
        filters=32,
        kernel_size=3,
        padding="same",
        activation="relu"
    )(inputs)

    x = layers.SeparableConv1D(
        filters=32,
        kernel_size=3,
        padding="same",
        activation="relu"
    )(x)

    x = layers.GlobalAveragePooling1D()(x)


    # Hazard-specific heads

    flood_head = layers.Dense(
        16,
        activation="relu",
        name="flood_features"
    )(x)

    flood_output = layers.Dense(
        3,
        activation="softmax",
        name="flood_output"
    )(flood_head)


    fire_head = layers.Dense(
        16,
        activation="relu",
        name="fire_features"
    )(x)

    fire_output = layers.Dense(
        3,
        activation="softmax",
        name="fire_output"
    )(fire_head)


    landslide_head = layers.Dense(
        16,
        activation="relu",
        name="landslide_features"
    )(x)

    landslide_output = layers.Dense(
        3,
        activation="softmax",
        name="landslide_output"
    )(landslide_head)


    model = Model(
        inputs=inputs,
        outputs=[
            flood_output,
            fire_output,
            landslide_output
        ]
    )

    # Model Compilation with explicit multi-output losses and metrics
    model.compile(
        optimizer="adam",
        loss={
            "flood_output": "sparse_categorical_crossentropy",
            "fire_output": "sparse_categorical_crossentropy",
            "landslide_output": "sparse_categorical_crossentropy"
        },
        metrics={
            "flood_output": ["accuracy"],
            "fire_output": ["accuracy"],
            "landslide_output": ["accuracy"]
        }
    )

    return model

In [ ]:
# 7. Prototype Training

print("\nGenerating demo data...")
X_train, y_train_dict = generate_data()

print("Input data shape:", X_train.shape)
print("Flood labels shape:", y_train_dict["flood"].shape)
print("Fire labels shape:", y_train_dict["fire"].shape)
print("Landslide labels shape:", y_train_dict["landslide"].shape)

model = build_model()

print("\nTraining lightweight prototype...\n")

history = model.fit(
    X_train,
    {
        "flood_output": y_train_dict["flood"],
        "fire_output": y_train_dict["fire"],
        "landslide_output": y_train_dict["landslide"]
    },
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)



Generating demo data...
Input data shape: (3000, 6, 8)
Flood labels shape: (3000,)
Fire labels shape: (3000,)
Landslide labels shape: (3000,)

Training lightweight prototype...

Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 12s 219ms/step - fire_output_accuracy: 0.9271 - fire_output_loss: 0.3028 - flood_output_accuracy: 0.9529 - flood_output_loss: 0.2313 - landslide_output_accuracy: 0.9758 - landslide_output_loss: 0.2028 - loss: 0.7388 - val_fire_output_accuracy: 0.9650 - val_fire_output_loss: 0.1695 - val_flood_output_accuracy: 0.9567 - val_flood_output_loss: 0.1883 - val_landslide_output_accuracy: 0.9700 - val_landslide_output_loss: 0.1904 - val_loss: 0.5117
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - fire_output_accuracy: 0.9613 - fire_output_loss: 0.1491 - flood_output_accuracy: 0.9529 - flood_output_loss: 0.1866 - landslide_output_accuracy: 0.9758 - landslide_output_loss: 0.1344 - loss: 0.4694 - val_fire_output_accuracy: 0.9650 - val_fire_output_loss: 0.1475 - val_flood_output_accu

In [ ]:
# 8. Training/Evaluation Summary

# Print final training metrics
print("\n===== Final Training Metrics =====")
for metric_name in history.history.keys():
    if "val_" not in metric_name: # Only consider training metrics for summary here
        print(f"{metric_name}: {history.history[metric_name][-1]:.4f}")

# Note: For a real project, you would save the model and evaluate it on a separate test set.
# For this prototype, the validation split during training serves as a basic evaluation.


===== Final Training Metrics =====
fire_output_accuracy: 0.9613
fire_output_loss: 0.1075
flood_output_accuracy: 0.9529
flood_output_loss: 0.1705
landslide_output_accuracy: 0.9758
landslide_output_loss: 0.1073
loss: 0.3817


## 8. Common Utility Functions

In [ ]:
import numpy as np
import pandas as pd
import io # Needed for parsing string data in demo scenarios

def get_state(class_index):
    """
    Converts a predicted class index (0, 1, 2) into a hazard state (NORMAL, WATCH, ALERT).
    """
    if class_index == 0:
        return "NORMAL"
    elif class_index == 1:
        return "WATCH"
    elif class_index == 2:
        return "ALERT"
    else:
        return "UNKNOWN" # Should not happen with valid input

def validate_sensor_value(feature_name, value):
    """
    Validates a single sensor value against its predefined range.
    Returns (True, None) if valid, or (False, error_message) if invalid.
    """
    if feature_name not in SENSOR_RANGES:
        return False, f"Unknown feature: {feature_name}."

    min_val, max_val = SENSOR_RANGES[feature_name]

    if not isinstance(value, (int, float)):
        return False, f"Invalid input type. Please enter a numeric value for {feature_name}."

    if not (min_val <= value <= max_val):
        return False, f"Invalid value. {feature_name} must be between {min_val} and {max_val}."

    return True, None

def process_window(readings_list):
    """
    Converts a list of 6x8 readings into a (1, 6, 8) numpy array for the model.
    Performs validation on shape and data type.
    """
    if not isinstance(readings_list, list) or len(readings_list) != WINDOW_SIZE:
        raise ValueError(f"Input must be a list of {WINDOW_SIZE} readings.")

    processed_readings = []
    for i, reading in enumerate(readings_list):
        if not isinstance(reading, list) or len(reading) != FEATURES:
            raise ValueError(f"Reading {i+1} must contain exactly {FEATURES} features.")
        try:
            processed_readings.append([float(val) for val in reading])
        except ValueError:
            raise ValueError(f"Reading {i+1} contains non-numeric values. All values must be numeric.")

    input_array = np.array(processed_readings, dtype=np.float32)

    if input_array.shape != (WINDOW_SIZE, FEATURES):
        raise ValueError(f"Internal error: Expected shape ({WINDOW_SIZE}, {FEATURES}), but got {input_array.shape}.")

    # Add batch dimension
    return np.expand_dims(input_array, axis=0)

def display_input_window(input_data_2d, title="30-MINUTE INPUT WINDOW"):
    """
    Displays the 2D input window (6x8) in a formatted table.
    """
    feature_names = [
        "Rainfall", "Temperature", "Humidity", "Wind Speed",
        "Water Level", "Soil Moisture", "Slope/Tilt", "Vibration"
    ]
    df_display = pd.DataFrame(input_data_2d, columns=feature_names)
    df_display.index = [f"Step {k+1}" for k in range(WINDOW_SIZE)]

    print(f"\n{'='*40}")
    print(f"{title.center(40)}")
    print(f"{('='*40)}")
    display(df_display)
    print(f"\nModel input shape: {np.expand_dims(input_data_2d, axis=0).shape}")

def predict_hazards(input_window_batch, model):
    """
    Performs prediction using the model and returns formatted hazard states and confidence.
    input_window_batch should have shape (1, WINDOW_SIZE, FEATURES).
    """
    if input_window_batch.shape != (1, WINDOW_SIZE, FEATURES):
        raise ValueError(f"Input window for prediction must have shape (1, {WINDOW_SIZE}, {FEATURES}).")

    raw_predictions = model.predict(input_window_batch, verbose=0)

    results = {}
    hazard_types = ["flood", "fire", "landslide"]

    # raw_predictions is a list of arrays, one for each output head.
    # Each array will be of shape (1, 3) for the batch size of 1 and 3 classes.
    for i, hazard_type in enumerate(hazard_types):
        probs = raw_predictions[i][0] # Get probabilities for the current hazard (shape will be (3,))
        predicted_class_index = np.argmax(probs)
        confidence = probs[predicted_class_index] * 100 # Convert to percentage

        results[hazard_type] = {
            "class_index": int(predicted_class_index),
            "state": get_state(predicted_class_index),
            "confidence": float(confidence)
        }
    return results

def display_prediction_output(results, scenario_title="MULTI-HAZARD PROTOTYPE PREDICTION"):
    """
    Displays the hazard prediction results in a formatted way.
    """
    print(f"\n{'='*40}")
    print(f"{scenario_title.center(40)}")
    print(f"{('='*40)}")

    for hazard_type in ["flood", "fire", "landslide"]:
        print(f"\n{hazard_type.upper()}")
        print(f"State      : {results[hazard_type]['state']}")
        print(f"Confidence : {results[hazard_type]['confidence']:.1f}%") # Display confidence with 1 decimal place

    print(f"\n{'='*40}")
    print(f"NOTE: These are prototype predictions. They are not scientifically validated")
    print(f"      and should not be used for real-world decision-making.")
    print(f"{('='*40)}")

## 9. Predefined Demo Scenarios

In [ ]:
# Demo raw data strings for each scenario
raw_input_data_normal = """
10,25,70,5,2.0,40,10,10
10,25,70,5,2.0,40,10,10
10,25,70,5,2.0,40,10,10
10,25,70,5,2.0,40,10,10
10,25,70,5,2.0,40,10,10
10,25,70,5,2.0,40,10,10
"""

raw_input_data_flood = """
20,30,70,10,4.5,50,15,20
30,31,68,12,4.9,55,16,22
40,31,65,14,5.4,60,18,25
50,32,62,16,6.0,68,20,30
60,32,58,18,6.8,75,22,35
75,33,55,20,7.6,82,24,40
"""

raw_input_data_fire = """
5,30,65,10,3.0,30,10,10
4,32,60,15,3.0,28,10,12
3,34,55,20,3.0,25,11,15
2,36,48,25,3.0,22,11,18
1,38,42,30,3.0,20,12,20
0,41,35,38,3.0,18,12,22
"""

raw_input_data_landslide = """
10,28,70,10,4.0,45,15,15
20,29,68,12,4.1,52,18,20
30,29,65,14,4.2,60,21,28
40,30,62,16,4.3,68,25,40
55,30,58,18,4.4,77,30,58
70,31,55,20,4.5,86,36,78
"""

def run_predefined_scenario(choice, model):
    """
    Runs a predefined multi-hazard scenario based on the given choice.
    Displays input window and model predictions.
    """
    selected_raw_data = ""
    scenario_title = ""

    if choice == 1:
        selected_raw_data = raw_input_data_normal
        scenario_title = "Normal Conditions"
    elif choice == 2:
        selected_raw_data = raw_input_data_flood
        scenario_title = "Increasing Flood Conditions"
    elif choice == 3:
        selected_raw_data = raw_input_data_fire
        scenario_title = "Increasing Forest Fire Conditions"
    elif choice == 4:
        selected_raw_data = raw_input_data_landslide
        scenario_title = "Increasing Landslide Conditions"
    else:
        print("Invalid scenario choice provided to function. Please choose between 1 and 4.")
        return

    print(f"\nProcessing predefined scenario: {scenario_title}")

    try:
        # Use io.StringIO to treat the string as a file
        readings_2d = np.genfromtxt(io.StringIO(selected_raw_data), delimiter=',', dtype=np.float32)

        # Process the window using the utility function
        input_window_for_model = process_window(readings_2d.tolist()) # Convert back to list for process_window validation

        # Display the input window and model input shape
        display_input_window(readings_2d, title=f"TEST SCENARIO: {scenario_title.upper()}")

        # Get predictions
        hazard_predictions = predict_hazards(input_window_for_model, model)

        # Display predictions
        display_prediction_output(hazard_predictions, scenario_title=f"PREDICTION FOR {scenario_title.upper()}")

    except ValueError as e:
        print(f"Error in scenario processing: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

## 10. Manual ESP32 Input Simulator

In [ ]:
def run_manual_esp32_simulator(model):
    """
    Simulates ESP32 sensor input, validates readings, and performs model prediction.
    """
    print(f"\n{'='*40}\n{'REAL-TIME ESP32 INPUT SIMULATOR'.center(40)}\n{('='*40)}")
    print("Enter six consecutive sensor readings for the latest 30-minute window.")
    print(f"Each reading should have {FEATURES} features. Robust input validation will be performed.")

    manual_readings_list = []
    feature_names = [
        "Rainfall", "Temperature", "Humidity", "Wind Speed",
        "Water Level", "Soil Moisture", "Slope/Tilt", "Vibration"
    ]

    for i in range(WINDOW_SIZE):
        print(f"\n--- Reading {i+1} of {WINDOW_SIZE} ---")
        current_reading_features = []
        for j, feature_name in enumerate(feature_names):
            while True:
                try:
                    value_str = input(f"{feature_name} (Range: {SENSOR_RANGES[feature_name][0]}-{SENSOR_RANGES[feature_name][1]}): ")
                    value = float(value_str)

                    # Validate the input using the new utility function
                    is_valid, error_msg = validate_sensor_value(feature_name, value)
                    if is_valid:
                        current_reading_features.append(value)
                        break
                    else:
                        print(f"Invalid input: {error_msg}. Please try again.")
                except ValueError:
                    print("Invalid input. Please enter a numeric value.")
        manual_readings_list.append(current_reading_features)

    print(f"\n{'='*40}\n{'ESP32 SENSOR WINDOW'.center(40)}\n{('='*40)}")
    print("6 readings collected successfully.")

    try:
        # Process the manual input using the utility function
        input_window_for_model_manual = process_window(manual_readings_list)

        # Display the input window in a table format
        display_input_window(np.array(manual_readings_list), title="MANUAL ESP32 INPUT WINDOW")

        # Explain the sliding window concept
        print(f"\n{'='*40}\n{'SLIDING WINDOW EXPLANATION'.center(40)}\n{('='*40)}")
        print("The six readings above represent the current 30-minute window (R1 R2 R3 R4 R5 R6).")
        print("In a real-time system, when a new ESP32 reading (R7) arrives, the oldest reading (R1) is dropped.")
        print("The window then shifts to (R2 R3 R4 R5 R6 R7), maintaining a consistent 30-minute view.")
        print(f"\nThis manual input system is an ESP32 simulator. The real system flow will be:")
        print(f"ESP32 -> New sensor reading (R7) -> Store reading -> Maintain latest 6 readings ->")
        print(f"30-minute sliding window -> Existing model -> Flood / Fire / Landslide prediction.")
        print(f"{('='*40)}")

        # Get predictions
        hazard_predictions_manual = predict_hazards(input_window_for_model_manual, model)

        # Display predictions
        display_prediction_output(hazard_predictions_manual, scenario_title="MULTI-HAZARD PREDICTION (MANUAL INPUT)")

    except ValueError as e:
        print(f"Error processing manual input: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

## 11. Test Scenario Menu

In [ ]:
# 11. Test Scenario Menu

def run_interactive_menu(model):
    while True:
        print(f"\n{'='*40}")
        print(f"{'SELECT TEST SCENARIO'.center(40)}")
        print(f"{('='*40)}")
        print("1 -> Normal conditions")
        print("2 -> Increasing flood conditions")
        print("3 -> Increasing forest-fire conditions")
        print("4 -> Increasing landslide conditions")
        print("5 -> Manual ESP32 sensor input")
        print("0 -> Exit demo")
        print(f"{('='*40)}")

        try:
            choice = int(input("Enter your choice (0-5): "))

            if choice == 0:
                print("Exiting demo. Goodbye!")
                break
            elif 1 <= choice <= 4:
                run_predefined_scenario(choice, model)
            elif choice == 5:
                run_manual_esp32_simulator(model)
            else:
                print("Invalid choice. Please enter a number between 0 and 5.")
        except ValueError:
            print("Invalid input. Please enter a number.")
        except Exception as e:
            print(f"An error occurred during scenario execution: {e}")

# Run the interactive menu
run_interactive_menu(model)


          SELECT TEST SCENARIO          
1 -> Normal conditions
2 -> Increasing flood conditions
3 -> Increasing forest-fire conditions
4 -> Increasing landslide conditions
5 -> Manual ESP32 sensor input
0 -> Exit demo
Enter your choice (0-5): 0
Exiting demo. Goodbye!


## 12. Final Prototype Output

This notebook demonstrates a complete `input -> processing -> prediction` flow for a multi-hazard early warning prototype. It includes:

*   **Synthetic Data Generation:** Generates data with physically sensible rules and balanced class distributions (NORMAL, WATCH, ALERT).
*   **Lightweight Model:** A `SeparableConv1D` model designed for efficient edge deployment.
*   **Input Validation:** Robust validation for manual sensor inputs using predefined ranges.
*   **Demo Scenarios:** Predefined scenarios (Normal, Flood, Fire, Landslide) and an interactive ESP32 simulator.
*   **Multi-Class Output:** Model predictions show the hazard state (NORMAL/WATCH/ALERT) and confidence percentage for each hazard.
*   **Sliding Window Logic:** Demonstrates the 30-minute sliding window concept for continuous monitoring.

While this prototype effectively showcases the system flow and user interaction, it is built with synthetic data and is not scientifically validated for real-world deployment. Further research, real-world data integration, and rigorous testing would be required for a production-ready system.

Thank you for reviewing the Multi-Hazard Prediction Prototype!